# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² regression dataset using the `mlcroissant` library, following the Croissant schema protocol and always referring to data elements by their `@id` fields.

### Dataset Source
The dataset source is published as a Croissant-compliant schema and accessible at:
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

We use the Croissant schema URL to initialize the dataset. The `mlcroissant.Dataset` object provides both high-level metadata and record access.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access and print the high-level metadata
print("\033[1m" + dataset.metadata.name + "\033[0m")
print(dataset.metadata.description)

## 2. Data Overview
List all available record sets and their fields, referencing them only by their `@id`s. This allows us to determine which record sets and fields are available for analysis.

We'll enumerate all record sets and print each one's `@id` and name, followed by their fields' `@id`s and names (if available).

In [ ]:
# List all record sets and their fields, using `@id`.
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets available in this dataset.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs.id}, name: {getattr(rs, 'name', 'N/A')}")
        for field in rs.fields:
            print(f"    Field @id: {field.id}, name: {getattr(field, 'name', 'N/A')}")

## 3. Data Extraction
To load and explore data, select a record set to extract (always referencing its `@id`).

**If the dataset has no record sets, this section will use available documentation/distributions as a demonstration.**

In [ ]:
# Prepare to extract dataframes by record set @id
dataframes = {}

# Collect all record set @ids
record_set_ids = [rs.id for rs in getattr(dataset, 'record_sets', [])]
print(f"Available record set @ids: {record_set_ids}")

# Extract records from each record set and load into a pandas DataFrame, reference by @id
for record_set_id in record_set_ids:
    print(f"Loading records for record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for {record_set_id}: {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"No records found for record set @id {record_set_id}.")

# If no dataframes loaded, inform the user; otherwise, pick the first one for demonstration
if not dataframes:
    print("No record set dataframes available for analysis.")
else:
    primary_record_set_id = next(iter(dataframes))  # Use the first available
    print(f"Proceeding with record set @id: {primary_record_set_id}")
    print(dataframes[primary_record_set_id].columns.tolist())
    display(dataframes[primary_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Example: filter, normalize, and group records using fields referenced by their `@id`.

We'll:
- Choose a numeric field by searching the DataFrame for numeric types.
- Demonstrate filtering, normalization, and grouping operations, all referencing fields by their `@id`.

**If no dataframes are loaded in the previous section, this cell will safely skip EDA.**

In [ ]:
# EDA only if data is available
import numpy as np

if dataframes:
    # Use the first available record set for demonstration
    record_set_id = primary_record_set_id
    df = dataframes[record_set_id]
    
    # Identify numeric fields by checking dtypes
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

    if not numeric_fields:
        print(f"No numeric fields found in record set @id {record_set_id}.")
    else:
        # Choose the first numeric field's `@id`
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field @id: {numeric_field_id}")

        # Set a sample threshold (here we use the mean)
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]

        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the selected numeric field
        field_norm = f"{numeric_field_id}_normalized"
        filtered_df[field_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, field_norm]].head())

        # Attempt to group by a non-numeric field (string/object type)
        group_fields = [col for col in df.columns if pd.api.types.is_string_dtype(df[col])]
        if group_fields:
            group_field_id = group_fields[0]
            print(f"Grouping by field @id: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            display(grouped_df.head())
        else:
            print("No group fields found for grouping operation.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize numeric field distributions or relationships between fields using their `@id`s. Example: histogram and scatter plot using the `@id` of the fields.

If data is missing, this cell will not execute.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_fields:
    # Histogram of the first numeric field
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id], kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Scatterplot if at least two numeric fields are available
    if len(numeric_fields) > 1:
        plt.figure(figsize=(6, 5))
        sns.scatterplot(x=df[numeric_fields[0]], y=df[numeric_fields[1]])
        plt.title(f"Scatter plot: {numeric_fields[0]} vs {numeric_fields[1]}")
        plt.xlabel(numeric_fields[0])
        plt.ylabel(numeric_fields[1])
        plt.show()
else:
    print("No numeric data available for visualization.")

## 6. Conclusion
This notebook demonstrated how to programmatically load, explore, and visualize data using the `mlcroissant` library, emphasizing referencing all dataset components (record sets, fields, and columns) by their `@id` fields as required by the Croissant schema.

_If you found no available record sets in the dataset, ensure the source provides tabular or structured record data. For more examples, refer to the [mlcroissant documentation](https://mlcroissant.github.io/)._